# 7. Create Training Sample And Paragraph Coding Sheet

This notebook samples parsed text files from `parsed_texts/publications_*` and turns that sample into a paragraph-level coding sheet for human labelling and ModernBERT prefilter training.


## Sample Parsed Texts

The sample is copied to `training_data/parsed_texts`; no raw PDFs are sampled here because PDF parsing is handled in notebook 6.


In [ ]:
# Resolve the repository and external data directory without machine-specific paths.
from pathlib import Path
import sys

for _candidate in (Path.cwd(), *Path.cwd().parents):
    if (_candidate / "project_config.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError("Run this notebook from within the cloned repository.")

from project_config import REPO_ROOT, DATA_ROOT, require_env

from pathlib import Path
import random
import re
import shutil

import pandas as pd

# =========================================================
# PATH CONFIG
# =========================================================
# Code is run from the OneDrive project code folder.
CODE_ROOT = REPO_ROOT
DATASET_CURATION_CODE_ROOT = CODE_ROOT / "dataset_curation"

# Data is read from and written to the share-drive project folder.
SHARE_ROOT = DATA_ROOT
# DATA_ROOT is configured through project_config.

PARSED_TEXT_ROOT = DATA_ROOT / "parsed_texts"
TRAINING_TEXT_SAMPLE_FOLDER = DATA_ROOT / "training_data" / "parsed_texts"
PARAGRAPH_CODING_FILE = DATA_ROOT / "training_data" / "paragraph_coding.csv"
LOOKUP_FILE = TRAINING_TEXT_SAMPLE_FOLDER / "training_set_dois.xlsx"

SAMPLE_SIZE = 3000
RANDOM_SEED = 44
OVERWRITE_EXISTING_SAMPLE = False

TRAINING_TEXT_SAMPLE_FOLDER.mkdir(parents=True, exist_ok=True)

# =========================================================
# COLLECT ALL PARSED TEXTS
# =========================================================
txt_folders = sorted(PARSED_TEXT_ROOT.glob("publications_*"))
all_txts = []

for folder in txt_folders:
    for txt_file in sorted(folder.glob("*.txt")):
        all_txts.append({
            "source_path": txt_file,
            "filename": txt_file.name,
            "source_folder": folder.name,
        })

print(f"Total parsed TXT files available: {len(all_txts):,}")

# =========================================================
# SAMPLE PARSED TEXTS
# =========================================================
random.seed(RANDOM_SEED)
sample_size_actual = min(SAMPLE_SIZE, len(all_txts))
sampled_txts = random.sample(all_txts, sample_size_actual)

print(f"Sampling {len(sampled_txts):,} TXT files")

# =========================================================
# COPY SAMPLE AND CREATE LOOKUP
# =========================================================
doi_lookup = []
skipped_existing = 0

for item in sampled_txts:
    src = item["source_path"]
    dst = TRAINING_TEXT_SAMPLE_FOLDER / item["filename"]

    if dst.exists() and not OVERWRITE_EXISTING_SAMPLE:
        skipped_existing += 1
    else:
        shutil.copy2(src, dst)

    safe_doi_name = re.sub(r" \(\d+\)$", "", src.stem)
    doi = safe_doi_name.replace("_", "/")

    doi_lookup.append({
        "filename": item["filename"],
        "doi": doi,
        "source_folder": item["source_folder"],
        "source_path": str(src),
    })

df_lookup = pd.DataFrame(doi_lookup)
df_lookup.to_excel(LOOKUP_FILE, index=False)

print(f"Copied or confirmed {len(sampled_txts):,} sampled TXT files in: {TRAINING_TEXT_SAMPLE_FOLDER}")
print(f"Skipped existing files: {skipped_existing:,}")
print(f"Saved DOI lookup table: {LOOKUP_FILE}")


## Split Sampled Texts Into Paragraphs

The output CSV is intended for human coding. After manual labelling, save the labelled version as `training_data/paragraph_coding_HT.xlsx` or update notebook 8's `TRAIN_FILE`.


In [ ]:
import re
import random
import pandas as pd
from pathlib import Path

# =========================================================
# SETTINGS
# =========================================================
input_folder = TRAINING_TEXT_SAMPLE_FOLDER
output_file = PARAGRAPH_CODING_FILE

drop_exact_duplicates_within_document = True
drop_boilerplate = False
drop_short_paragraphs = True

short_char_threshold = 40
short_word_threshold = 6
max_paragraph_chars = 1200

random_seed = 42
random.seed(random_seed)

# =========================================================
# PATTERNS
# =========================================================
BOILERPLATE_PATTERNS = [
    r"\bcreative commons\b",
    r"\bopen access\b",
    r"\blicen[cs]e\b",
    r"\bcopyright\b",
    r"\ball rights reserved\b",
    r"\bpublisher'?s note\b",
    r"\bsupplementary material\b",
    r"\badditional file\b",
    r"\bpreprint\b",
    r"\bmedrxiv\b",
    r"\bbiorxiv\b",
    r"\bthis article is distributed under\b",
    r"\bthe author\(s\)\b",
    r"\bpermission to share\b",
    r"\bcorresponding author\b",
]

SOURCE_SIGNAL_PATTERNS = [
    r"\bdata (?:were|was)?\s*(?:obtained|collected|drawn|retrieved|downloaded|taken|derived|sourced)\s+from\b",
    r"\busing data from\b",
    r"\bbased on data from\b",
    r"\bdata from\b",
    r"\bsource:? [A-Z]",
    r"\bdataset\b",
    r"\badministrative data\b",
    r"\bsurvey data\b",
    r"\bcensus\b",
    r"\bregistry\b",
    r"\bpanel data\b",
    r"\barchive\b",
    r"\bworld bank\b",
    r"\beurostat\b",
    r"\boecd\b",
    r"\bun data\b",
    r"\bwho\b",
]

# =========================================================
# HELPERS
# =========================================================
def clean_text(text):
    """Normalize whitespace while preserving paragraph breaks."""
    if pd.isna(text):
        return ""

    text = str(text)
    text = text.replace("\ufeff", " ")
    text = text.replace("\xa0", " ")
    text = text.replace("\u00ad", "")

    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n[ \t]+", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def recover_doi_from_filename(filename):
    stem = Path(filename).stem
    return stem.replace("_", "/")


def is_probable_heading(text):
    t = text.strip()
    if not t:
        return False

    if len(t) > 120:
        return False

    if re.search(r"[.!?]$", t):
        return False

    if re.match(r"^\d+(\.\d+)*\s+[A-Z][A-Za-z].*$", t):
        return True

    words = re.findall(r"[A-Za-z][A-Za-z\-]*", t)
    if not words:
        return False

    if t.isupper() and len(words) <= 12:
        return True

    capitalized_ratio = sum(w[0].isupper() for w in words) / len(words)
    if len(words) <= 10 and capitalized_ratio >= 0.8:
        return True

    return False


def is_boilerplate(text):
    t = text.lower()
    return any(re.search(pattern, t) for pattern in BOILERPLATE_PATTERNS)


def has_source_signal(text):
    t = text.lower()
    return any(re.search(pattern, t) for pattern in SOURCE_SIGNAL_PATTERNS)


def normalize_for_dedupe(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text


def count_words(text):
    return len(re.findall(r"\b\w+\b", text))


def is_short_paragraph(text):
    return len(text) < short_char_threshold or count_words(text) < short_word_threshold


def merge_wrapped_lines_into_paragraphs(text):
    """
    Reconstruct paragraphs from parsed PDF text.
    """
    lines = text.split("\n")
    paragraphs = []
    current = ""

    for raw_line in lines:
        line = raw_line.strip()

        if not line:
            if current.strip():
                paragraphs.append(current.strip())
                current = ""
            continue

        if is_probable_heading(line):
            if current.strip():
                paragraphs.append(current.strip())
                current = ""
            paragraphs.append(line)
            continue

        if not current:
            current = line
            continue

        if current.endswith("-") and re.match(r"^[a-z]", line):
            current = current[:-1] + line
        else:
            current = f"{current} {line}"

    if current.strip():
        paragraphs.append(current.strip())

    return paragraphs


def split_long_paragraph(paragraph, max_chars=None):
    """
    Optionally split very long blocks into smaller chunks.
    """
    if not max_chars or len(paragraph) <= max_chars:
        return [paragraph]

    sentences = re.split(r"(?<=[.!?])\s+(?=[A-Z])", paragraph)
    if len(sentences) <= 1:
        return [paragraph]

    chunks = []
    current = ""

    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue

        if not current:
            current = sentence
            continue

        candidate = f"{current} {sentence}"
        if len(candidate) <= max_chars:
            current = candidate
        else:
            chunks.append(current.strip())
            current = sentence

    if current.strip():
        chunks.append(current.strip())

    return chunks


def split_into_paragraphs(text):
    text = clean_text(text)
    if not text:
        return []

    paragraphs = merge_wrapped_lines_into_paragraphs(text)

    final_paragraphs = []
    for p in paragraphs:
        p = clean_text(p)
        if not p:
            continue
        final_paragraphs.extend(split_long_paragraph(p, max_paragraph_chars))

    return [p for p in final_paragraphs if p.strip()]


# =========================================================
# BUILD DATASET: ONE RANDOM PARAGRAPH PER DOCUMENT
# =========================================================
rows = []

txt_files = sorted(input_folder.glob("*.txt"))

documents_processed = 0
documents_with_no_kept_paragraph = 0
duplicates_removed = 0
boilerplate_flagged = 0
short_flagged = 0
heading_flagged = 0
source_signal_flagged = 0

for file_path in txt_files:
    txt_file = file_path.name

    try:
        text = file_path.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        text = file_path.read_text(encoding="latin-1")

    text = clean_text(text)
    raw_paragraphs = split_into_paragraphs(text)
    doi = recover_doi_from_filename(txt_file)

    seen_in_document = set()
    kept_paragraphs = []

    for original_idx, paragraph in enumerate(raw_paragraphs, start=1):
        paragraph = clean_text(paragraph)
        if not paragraph:
            continue

        paragraph_norm = normalize_for_dedupe(paragraph)
        if drop_exact_duplicates_within_document and paragraph_norm in seen_in_document:
            duplicates_removed += 1
            continue
        seen_in_document.add(paragraph_norm)

        word_count = count_words(paragraph)
        char_count = len(paragraph)
        heading_like = is_probable_heading(paragraph)
        boilerplate = is_boilerplate(paragraph)
        short_para = is_short_paragraph(paragraph)
        source_signal = has_source_signal(paragraph)

        if boilerplate:
            boilerplate_flagged += 1
        if short_para:
            short_flagged += 1
        if heading_like:
            heading_flagged += 1
        if source_signal:
            source_signal_flagged += 1

        if drop_boilerplate and boilerplate:
            continue

        if drop_short_paragraphs and short_para and not source_signal:
            continue

        kept_paragraphs.append({
            "original_paragraph_nr": original_idx,
            "text": paragraph,
            "word_count": word_count,
            "char_count": char_count,
            "is_short": short_para,
            "is_boilerplate": boilerplate,
            "is_heading_like": heading_like,
            "has_source_signal": source_signal,
        })

    if not kept_paragraphs:
        documents_with_no_kept_paragraph += 1
        documents_processed += 1
        continue

    chosen = random.choice(kept_paragraphs)

    rows.append({
        "paragraph_id": f"{file_path.stem}_p1",
        "document_id": file_path.stem,
        "doi": doi,
        "paragraph_nr": 1,
        "text": chosen["text"],
        "is_boilerplate": chosen["is_boilerplate"],
        "has_source_signal": chosen["has_source_signal"],
        "label": "",
        "coder_notes": ""
    })

    documents_processed += 1

df_paragraphs = pd.DataFrame(rows)

df_paragraphs = df_paragraphs.sort_values(
    by=["document_id"]
).reset_index(drop=True)

output_file.parent.mkdir(parents=True, exist_ok=True)
df_paragraphs.to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"Documents processed: {documents_processed}")
print(f"Documents with 1 randomly selected saved paragraph: {len(df_paragraphs)}")
print(f"Documents with no kept paragraph: {documents_with_no_kept_paragraph}")
print(f"Exact duplicates removed within document: {duplicates_removed}")
print(f"Flagged as boilerplate: {boilerplate_flagged}")
print(f"Flagged as short: {short_flagged}")
print(f"Flagged as heading-like: {heading_flagged}")
print(f"Flagged as possible source-signal: {source_signal_flagged}")
print(f"Saved: {output_file}")